## 1. Configuração do Spark e Delta Lake

In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("Trabalho Engenharia de Dados - Delta Lake")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark:", spark.version)

:: loading settings :: url = jar:file:/home/jeanvv/.cache/pypoetry/virtualenvs/spark-iceberg-tij3t2TE-py3.11/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jeanvv/.ivy2/cache
The jars for the packages stored in: /home/jeanvv/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e89505c8-6164-4fe0-849d-c4f4b23353b8;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.3 in central
	found io.delta#delta-storage;3.3.3 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 406ms :: artifacts dl 17ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.3 from central in [default]
	io.delta#delta-storage;3.3.3 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0

Spark: 3.5.3


## 2. Carregamento e análise do dataset

In [2]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../../data/metacritic_dataset_clean_sep_2025.csv")
)

## 3. Tratamento dos dados

In [3]:
df.printSchema()
df.show(5)
print("Registros:", df.count())

root
 |-- name: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- metascore: double (nullable = true)
 |-- user_score: double (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- genre: string (nullable = true)

+--------------------+-----------------+------------+---------+----------+--------------+--------------------+-----------------+
|                name|         platform|release_date|metascore|user_score|     developer|           publisher|            genre|
+--------------------+-----------------+------------+---------+----------+--------------+--------------------+-----------------+
|The Legend of Zel...|      Nintendo 64|  1998-11-23|     99.0|       9.1|      Nintendo|['Nintendo', 'Gra...|Open-World Action|
|         SoulCalibur|        Dreamcast|  1999-09-08|     98.0|       7.6|         Namco|           ['Namco']|      3D Fighting|
|         SoulCalibur|iOS (iP

In [4]:
df.groupBy("name", "platform", "release_date") \
    .count() \
    .filter("count > 1") \
    .orderBy("count", ascending=False) \
    .show(20)

[Stage 6:>                                                          (0 + 1) / 1]

+--------------------+-----------------+------------+-----+
|                name|         platform|release_date|count|
+--------------------+-----------------+------------+-----+
|BioShock Infinite...|    PlayStation 3|  2013-11-12|    2|
|BioShock Infinite...|               PC|  2013-11-12|    2|
|Fatal Frame: Spec...|             Xbox|  2002-11-22|    2|
|      The Descendant|               PC|  2016-03-24|    2|
|     Death Come True|               PC|  2020-06-24|    2|
|Dungelot: Shatter...|iOS (iPhone/iPad)|  2016-02-18|    2|
|Legacy of Kain: D...|             Xbox|  2003-11-11|    2|
|Her Majesty's SPI...|    PlayStation 4|  2016-12-07|    2|
|Hot Wheels Unleashed|               PC|  2021-09-27|    2|
|   Crazy Taxi (1999)|        Dreamcast|  2010-11-24|    2|
|             DOOM 64|    PlayStation 4|  2020-03-20|    2|
|   Old Man's Journey|  Nintendo Switch|  2017-05-17|    2|
|       John Wick Hex|  Nintendo Switch|  2019-10-08|    2|
|             Minoria|  Nintendo Switch|

In [5]:
df = df.dropDuplicates()

print("Registros após remoção de duplicatas:", df.count())

[Stage 9:>                                                          (0 + 1) / 1]

Registros após remoção de duplicatas: 21913


In [6]:
df.groupBy("name", "platform", "release_date") \
    .count() \
    .filter("count > 1") \
    .orderBy("count", ascending=False) \
    .show(20, truncate=False)

[Stage 15:>                                                         (0 + 1) / 1]

+---------------------------------------------------+-----------------+------------+-----+
|name                                               |platform         |release_date|count|
+---------------------------------------------------+-----------------+------------+-----+
|Osmos                                              |iOS (iPhone/iPad)|2009-08-18  |2    |
|Transformers: Revenge of the Fallen The Mobile Game|DS               |2009-06-23  |2    |
|Transformers: The Game                             |DS               |2007-06-26  |2    |
+---------------------------------------------------+-----------------+------------+-----+



In [7]:
from pyspark.sql.functions import monotonically_increasing_id

df = df.withColumn(
    "game_id",
    monotonically_increasing_id()
)

df.select(
    "game_id",
    "name",
    "platform",
    "release_date",
    "metascore",
    "user_score"
).show(10, truncate=False)

+-------+-----------------------------+----------------+------------+---------+----------+
|game_id|name                         |platform        |release_date|metascore|user_score|
+-------+-----------------------------+----------------+------------+---------+----------+
|0      |The Elder Scrolls V: Skyrim  |Xbox 360        |2011-11-11  |96.0     |8.6       |
|1      |Braid                        |PC              |2008-08-06  |90.0     |8.5       |
|2      |Tom Clancy's Splinter Cell   |Game Boy Advance|2002-11-17  |77.0     |6.5       |
|3      |NBA 2K13                     |PSP             |2012-10-02  |85.0     |7.3       |
|4      |Guilty Gear -Strive-         |PlayStation 5   |2021-06-08  |87.0     |7.9       |
|5      |Kathy Rain: Director's Cut   |PC              |2021-10-26  |87.0     |7.0       |
|6      |Monster Hunter Rise: Sunbreak|Xbox Series X   |2022-06-30  |85.0     |7.8       |
|7      |Bastion                      |Nintendo Switch |2011-07-20  |90.0     |7.7       |

In [8]:
print("Total de registros:", df.count())
print("IDs distintos:", df.select("game_id").distinct().count())

Total de registros: 21913
IDs distintos: 21913


## 4. Criação da tabela Delta

In [9]:
import os

delta_path = os.path.abspath("../../data/delta/metacritic_games")

print("Caminho da tabela Delta:")
print(delta_path)

Caminho da tabela Delta:
/home/jeanvv/Faculdade/2026-2/Engenharia de Dados/trabalho-eng-dados-apache-spark/data/delta/metacritic_games


In [10]:
df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(delta_path)

26/10/07 18:28:25 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

In [11]:
df_delta = (
    spark.read
    .format("delta")
    .load(delta_path)
)

df_delta.show(5, truncate=False)

print("Registros:", df_delta.count())

+---------------------------+----------------+------------+---------+----------+---------------------+----------------------------------------------+----------------+-------+---------------+
|name                       |platform        |release_date|metascore|user_score|developer            |publisher                                     |genre           |game_id|rating_category|
+---------------------------+----------------+------------+---------+----------+---------------------+----------------------------------------------+----------------+-------+---------------+
|The Elder Scrolls V: Skyrim|Xbox 360        |2011-11-11  |96.0     |8.6       |Bethesda Game Studios|['Bethesda Softworks']                        |Western RPG     |0      |NULL           |
|Braid                      |PC              |2008-08-06  |90.0     |8.5       |Number None Inc.     |['Number None Inc.', 'Microsoft Game Studios']|2D Platformer   |1      |NULL           |
|Tom Clancy's Splinter Cell |Game Boy Advance

In [12]:
from delta.tables import DeltaTable

print(
    "É uma tabela Delta:",
    DeltaTable.isDeltaTable(spark, delta_path)
)

É uma tabela Delta: True


In [13]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS metacritic_games
    USING DELTA
    LOCATION '{delta_path}'
""")

DataFrame[]

In [14]:
spark.sql("SHOW TABLES").show()

+---------+----------------+-----------+
|namespace|       tableName|isTemporary|
+---------+----------------+-----------+
|  default|metacritic_games|      false|
+---------+----------------+-----------+



In [15]:
spark.sql("""
    SELECT *
    FROM metacritic_games
    LIMIT 5
""").show(truncate=False)

+---------------------------+----------------+------------+---------+----------+---------------------+----------------------------------------------+----------------+-------+---------------+
|name                       |platform        |release_date|metascore|user_score|developer            |publisher                                     |genre           |game_id|rating_category|
+---------------------------+----------------+------------+---------+----------+---------------------+----------------------------------------------+----------------+-------+---------------+
|The Elder Scrolls V: Skyrim|Xbox 360        |2011-11-11  |96.0     |8.6       |Bethesda Game Studios|['Bethesda Softworks']                        |Western RPG     |0      |NULL           |
|Braid                      |PC              |2008-08-06  |90.0     |8.5       |Number None Inc.     |['Number None Inc.', 'Microsoft Game Studios']|2D Platformer   |1      |NULL           |
|Tom Clancy's Splinter Cell |Game Boy Advance

In [16]:
spark.sql("""
    DESCRIBE HISTORY metacritic_games
""").show(truncate=False)

+-------+-----------------------+------+--------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+--------+---------+-----------+-----------------+-------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+----------

In [17]:
from pyspark.sql.functions import to_date

novo_jogo = spark.createDataFrame([
    (
        "Delta Lake Demo Game",
        "PC",
        "2026-10-07",
        85.0,
        8.5,
        "Demo Studio",
        "Demo Publisher",
        "Action"
    )
], [
    "name",
    "platform",
    "release_date",
    "metascore",
    "user_score",
    "developer",
    "publisher",
    "genre"
]).withColumn(
    "release_date",
    to_date("release_date")
)

novo_jogo.show()

+--------------------+--------+------------+---------+----------+-----------+--------------+------+
|                name|platform|release_date|metascore|user_score|  developer|     publisher| genre|
+--------------------+--------+------------+---------+----------+-----------+--------------+------+
|Delta Lake Demo Game|      PC|  2026-10-07|     85.0|       8.5|Demo Studio|Demo Publisher|Action|
+--------------------+--------+------------+---------+----------+-----------+--------------+------+



In [18]:
novo_jogo.show()

+--------------------+--------+------------+---------+----------+-----------+--------------+------+
|                name|platform|release_date|metascore|user_score|  developer|     publisher| genre|
+--------------------+--------+------------+---------+----------+-----------+--------------+------+
|Delta Lake Demo Game|      PC|  2026-10-07|     85.0|       8.5|Demo Studio|Demo Publisher|Action|
+--------------------+--------+------------+---------+----------+-----------+--------------+------+



In [19]:
novo_jogo.createOrReplaceTempView("novo_jogo")

In [20]:
spark.table("metacritic_games").printSchema()

root
 |-- name: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- metascore: double (nullable = true)
 |-- user_score: double (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- game_id: long (nullable = true)
 |-- rating_category: string (nullable = true)



In [21]:
spark.table("metacritic_games").printSchema()

root
 |-- name: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- metascore: double (nullable = true)
 |-- user_score: double (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- game_id: long (nullable = true)
 |-- rating_category: string (nullable = true)



In [22]:
spark.sql("""
    SELECT MAX(game_id) AS maior_game_id
    FROM metacritic_games
""").show()

+-------------+
|maior_game_id|
+-------------+
|   8589946390|
+-------------+



## 5. INSERT

In [23]:
spark.sql("""
    INSERT INTO metacritic_games (
        name,
        platform,
        release_date,
        metascore,
        user_score,
        developer,
        publisher,
        genre,
        game_id
    )
    SELECT
        name,
        platform,
        release_date,
        metascore,
        user_score,
        developer,
        publisher,
        genre,
        (SELECT MAX(game_id) + 1 FROM metacritic_games)
    FROM novo_jogo
""")

DataFrame[]

In [24]:
spark.sql("""
    SELECT *
    FROM metacritic_games
    WHERE name = 'Delta Lake Demo Game'
""").show(truncate=False)

+--------------------+--------+------------+---------+----------+-----------+--------------+------+----------+---------------+
|name                |platform|release_date|metascore|user_score|developer  |publisher     |genre |game_id   |rating_category|
+--------------------+--------+------------+---------+----------+-----------+--------------+------+----------+---------------+
|Delta Lake Demo Game|PC      |2026-10-07  |85.0     |8.5       |Demo Studio|Demo Publisher|Action|8589946391|NULL           |
+--------------------+--------+------------+---------+----------+-----------+--------------+------+----------+---------------+



In [25]:
print("Registros:", spark.table("metacritic_games").count())

Registros: 21914


## 6. UPDATE

In [26]:
spark.sql("""
    UPDATE metacritic_games
    SET user_score = 9.0
    WHERE game_id = 8589946391
""")

DataFrame[num_affected_rows: bigint]

In [27]:
spark.sql("""
    SELECT
        name,
        platform,
        user_score,
        game_id
    FROM metacritic_games
    WHERE game_id = 8589946391
""").show(truncate=False)

+--------------------+--------+----------+----------+
|name                |platform|user_score|game_id   |
+--------------------+--------+----------+----------+
|Delta Lake Demo Game|PC      |9.0       |8589946391|
+--------------------+--------+----------+----------+



## 7. DELETE

In [28]:
spark.sql("""
    DELETE FROM metacritic_games
    WHERE game_id = 8589946391
""")

DataFrame[num_affected_rows: bigint]

In [29]:
spark.sql("""
    SELECT *
    FROM metacritic_games
    WHERE game_id = 8589946391
""").show(truncate=False)

+----+--------+------------+---------+----------+---------+---------+-----+-------+---------------+
|name|platform|release_date|metascore|user_score|developer|publisher|genre|game_id|rating_category|
+----+--------+------------+---------+----------+---------+---------+-----+-------+---------------+
+----+--------+------------+---------+----------+---------+---------+-----+-------+---------------+



## 8. Histórico e Time Travel

In [30]:
spark.sql("""
    DESCRIBE HISTORY metacritic_games
""").select(
    "version",
    "timestamp",
    "operation",
    "operationParameters"
).show(truncate=False)

+-------+-----------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation|operationParameters                                                                                                                                                                      |
+-------+-----------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|25     |2026-10-07 18:29:14.864|DELETE   |{predicate -> ["(game_id#5801L = 8589946391)"]}                                                                                                                                          |
|24     |2026-10-07 18:29:07.953|UPDATE   |{predicate -> ["(game_id#4295L = 8589

In [31]:
df_version_5 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 5)
    .load(delta_path)
)

df_version_5.filter(
    df_version_5.game_id == 8589946391
).show(truncate=False)

[Stage 141:================================================>      (44 + 4) / 50]

+--------------------+--------+------------+---------+----------+-----------+--------------+------+----------+
|name                |platform|release_date|metascore|user_score|developer  |publisher     |genre |game_id   |
+--------------------+--------+------------+---------+----------+-----------+--------------+------+----------+
|Delta Lake Demo Game|PC      |2026-10-07  |85.0     |9.0       |Demo Studio|Demo Publisher|Action|8589946391|
+--------------------+--------+------------+---------+----------+-----------+--------------+------+----------+



In [32]:
df_atual = (
    spark.read
    .format("delta")
    .load(delta_path)
)

df_atual.filter(
    df_atual.game_id == 8589946391
).show(truncate=False)

+----+--------+------------+---------+----------+---------+---------+-----+-------+---------------+
|name|platform|release_date|metascore|user_score|developer|publisher|genre|game_id|rating_category|
+----+--------+------------+---------+----------+---------+---------+-----+-------+---------------+
+----+--------+------------+---------+----------+---------+---------+-----+-------+---------------+



In [33]:
spark.sql("""
    SELECT *
    FROM metacritic_games VERSION AS OF 5
    WHERE game_id = 8589946391
""").show(truncate=False)

[Stage 147:==========================================>            (39 + 4) / 50]

+--------------------+--------+------------+---------+----------+-----------+--------------+------+----------+
|name                |platform|release_date|metascore|user_score|developer  |publisher     |genre |game_id   |
+--------------------+--------+------------+---------+----------+-----------+--------------+------+----------+
|Delta Lake Demo Game|PC      |2026-10-07  |85.0     |9.0       |Demo Studio|Demo Publisher|Action|8589946391|
+--------------------+--------+------------+---------+----------+-----------+--------------+------+----------+



## 9. MERGE

In [34]:
merge_data = spark.createDataFrame([
    (
        "Merge Demo Game",
        "PC",
        "2026-10-07",
        90.0,
        9.2,
        "Merge Studio",
        "Merge Publisher",
        "RPG",
        9999999999
    )
], [
    "name",
    "platform",
    "release_date",
    "metascore",
    "user_score",
    "developer",
    "publisher",
    "genre",
    "game_id"
]).withColumn(
    "release_date",
    to_date("release_date")
)

merge_data.show(truncate=False)

+---------------+--------+------------+---------+----------+------------+---------------+-----+----------+
|name           |platform|release_date|metascore|user_score|developer   |publisher      |genre|game_id   |
+---------------+--------+------------+---------+----------+------------+---------------+-----+----------+
|Merge Demo Game|PC      |2026-10-07  |90.0     |9.2       |Merge Studio|Merge Publisher|RPG  |9999999999|
+---------------+--------+------------+---------+----------+------------+---------------+-----+----------+



In [35]:
merge_data.createOrReplaceTempView("merge_data")

In [36]:
spark.sql("""
    MERGE INTO metacritic_games AS target
    USING merge_data AS source
    ON target.game_id = source.game_id

    WHEN MATCHED THEN
        UPDATE SET
            target.name = source.name,
            target.platform = source.platform,
            target.release_date = source.release_date,
            target.metascore = source.metascore,
            target.user_score = source.user_score,
            target.developer = source.developer,
            target.publisher = source.publisher,
            target.genre = source.genre

    WHEN NOT MATCHED THEN
        INSERT (
            name,
            platform,
            release_date,
            metascore,
            user_score,
            developer,
            publisher,
            genre,
            game_id
        )
        VALUES (
            source.name,
            source.platform,
            source.release_date,
            source.metascore,
            source.user_score,
            source.developer,
            source.publisher,
            source.genre,
            source.game_id
        )
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [37]:
spark.sql("""
    SELECT *
    FROM metacritic_games
    WHERE game_id = 9999999999
""").show(truncate=False)

+---------------+--------+------------+---------+----------+------------+---------------+-----+----------+---------------+
|name           |platform|release_date|metascore|user_score|developer   |publisher      |genre|game_id   |rating_category|
+---------------+--------+------------+---------+----------+------------+---------------+-----+----------+---------------+
|Merge Demo Game|PC      |2026-10-07  |90.0     |9.2       |Merge Studio|Merge Publisher|RPG  |9999999999|NULL           |
+---------------+--------+------------+---------+----------+------------+---------------+-----+----------+---------------+



In [38]:
merge_data_atualizado = spark.createDataFrame([
    (
        "Merge Demo Game",
        "PC",
        "2026-10-07",
        95.0,
        9.7,
        "Merge Studio",
        "Merge Publisher",
        "RPG",
        9999999999
    )
], [
    "name",
    "platform",
    "release_date",
    "metascore",
    "user_score",
    "developer",
    "publisher",
    "genre",
    "game_id"
]).withColumn(
    "release_date",
    to_date("release_date")
)

merge_data_atualizado.createOrReplaceTempView("merge_data_atualizado")

In [39]:
spark.sql("""
    MERGE INTO metacritic_games AS target
    USING merge_data_atualizado AS source
    ON target.game_id = source.game_id

    WHEN MATCHED THEN
        UPDATE SET
            target.metascore = source.metascore,
            target.user_score = source.user_score
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [40]:
spark.sql("""
    SELECT
        name,
        metascore,
        user_score,
        game_id
    FROM metacritic_games
    WHERE game_id = 9999999999
""").show()

+---------------+---------+----------+----------+
|           name|metascore|user_score|   game_id|
+---------------+---------+----------+----------+
|Merge Demo Game|     95.0|       9.7|9999999999|
+---------------+---------+----------+----------+



In [41]:
spark.sql("""
    DELETE FROM metacritic_games
    WHERE game_id = 9999999999
""")

DataFrame[num_affected_rows: bigint]

In [42]:
print("Registros:", spark.table("metacritic_games").count())

Registros: 21913


In [43]:
spark.sql("""
    SELECT *
    FROM metacritic_games
    WHERE game_id = 9999999999
""").show()

+----+--------+------------+---------+----------+---------+---------+-----+-------+---------------+
|name|platform|release_date|metascore|user_score|developer|publisher|genre|game_id|rating_category|
+----+--------+------------+---------+----------+---------+---------+-----+-------+---------------+
+----+--------+------------+---------+----------+---------+---------+-----+-------+---------------+



## 10. Schema Enforcement

In [44]:
schema_invalido = spark.createDataFrame([
    (
        "Schema Test Game",
        "PC",
        "2026-10-07",
        80.0,
        8.0,
        "Test Studio",
        "Test Publisher",
        "Action",
        8888888888,
        "coluna extra"
    )
], [
    "name",
    "platform",
    "release_date",
    "metascore",
    "user_score",
    "developer",
    "publisher",
    "genre",
    "game_id",
    "coluna_extra"
])

schema_invalido.printSchema()

root
 |-- name: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- metascore: double (nullable = true)
 |-- user_score: double (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- game_id: long (nullable = true)
 |-- coluna_extra: string (nullable = true)



In [45]:
try:
    (
        schema_invalido.write
        .format("delta")
        .mode("append")
        .save(delta_path)
    )
except Exception as e:
    print("Schema Enforcement bloqueou a operação:")
    print(type(e).__name__)
    print(e)

Schema Enforcement bloqueou a operação:
AnalysisException
[DELTA_FAILED_TO_MERGE_FIELDS] Failed to merge fields 'release_date' and 'release_date'


In [46]:
print("Registros:", spark.table("metacritic_games").count())

Registros: 21913


In [47]:
spark.table("metacritic_games").printSchema()

root
 |-- name: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- metascore: double (nullable = true)
 |-- user_score: double (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- game_id: long (nullable = true)
 |-- rating_category: string (nullable = true)



## 11. Schema Evolution

In [48]:
schema_evolution = spark.createDataFrame([
    (
        "Schema Evolution Game",
        "PC",
        "2026-10-07",
        88.0,
        8.8,
        "Evolution Studio",
        "Evolution Publisher",
        "RPG",
        7777777777,
        "Excelente"
    )
], [
    "name",
    "platform",
    "release_date",
    "metascore",
    "user_score",
    "developer",
    "publisher",
    "genre",
    "game_id",
    "rating_category"
]).withColumn(
    "release_date",
    to_date("release_date")
)

schema_evolution.printSchema()

root
 |-- name: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- metascore: double (nullable = true)
 |-- user_score: double (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- game_id: long (nullable = true)
 |-- rating_category: string (nullable = true)



In [49]:
(
    schema_evolution.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .save(delta_path)
)

In [50]:
spark.table("metacritic_games").printSchema()

root
 |-- name: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- metascore: double (nullable = true)
 |-- user_score: double (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- game_id: long (nullable = true)
 |-- rating_category: string (nullable = true)



In [51]:
spark.sql("""
    SELECT
        name,
        metascore,
        user_score,
        game_id,
        rating_category
    FROM metacritic_games
    WHERE game_id = 7777777777
""").show(truncate=False)

+---------------------+---------+----------+----------+---------------+
|name                 |metascore|user_score|game_id   |rating_category|
+---------------------+---------+----------+----------+---------------+
|Schema Evolution Game|88.0     |8.8       |7777777777|Excelente      |
+---------------------+---------+----------+----------+---------------+



In [52]:
spark.sql("""
    DESCRIBE HISTORY metacritic_games
""").select(
    "version",
    "timestamp",
    "operation",
    "operationParameters"
).show(truncate=False)

+-------+-----------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation|operationParameters                                                                                                                                                                      |
+-------+-----------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|29     |2026-10-07 18:29:46.893|WRITE    |{mode -> Append, partitionBy -> []}                                                                                                                                                      |
|28     |2026-10-07 18:29:42.011|DELETE   |{predicate -> ["(game_id#11807L = 999

## Time Travel

O Delta Lake mantém o histórico das alterações realizadas na tabela.
Isso permite consultar versões anteriores dos dados utilizando o número
da versão da tabela.

No exemplo abaixo, a versão 5 é consultada para recuperar o registro
que posteriormente foi excluído.

In [53]:
spark.sql("""
    SELECT
        name,
        metascore,
        user_score,
        game_id
    FROM metacritic_games VERSION AS OF 5
    WHERE game_id = 8589946391
""").show(truncate=False)

[Stage 235:==================================================>    (46 + 4) / 50]

+--------------------+---------+----------+----------+
|name                |metascore|user_score|game_id   |
+--------------------+---------+----------+----------+
|Delta Lake Demo Game|85.0     |9.0       |8589946391|
+--------------------+---------+----------+----------+



In [54]:
spark.sql("""
    SELECT
        name,
        metascore,
        user_score,
        game_id,
        rating_category
    FROM metacritic_games
    WHERE game_id = 8589946391
""").show(truncate=False)

+----+---------+----------+-------+---------------+
|name|metascore|user_score|game_id|rating_category|
+----+---------+----------+-------+---------------+
+----+---------+----------+-------+---------------+



## 12. Delta Log

In [55]:
import os

delta_log_path = os.path.join(delta_path, "_delta_log")

for arquivo in sorted(os.listdir(delta_log_path)):
    print(arquivo)

.00000000000000000000.crc.crc
.00000000000000000000.json.crc
.00000000000000000001.crc.crc
.00000000000000000001.json.crc
.00000000000000000002.crc.crc
.00000000000000000002.json.crc
.00000000000000000003.crc.crc
.00000000000000000003.json.crc
.00000000000000000004.crc.crc
.00000000000000000004.json.crc
.00000000000000000005.crc.crc
.00000000000000000005.json.crc
.00000000000000000006.crc.crc
.00000000000000000006.json.crc
.00000000000000000007.crc.crc
.00000000000000000007.json.crc
.00000000000000000008.crc.crc
.00000000000000000008.json.crc
.00000000000000000009.crc.crc
.00000000000000000009.json.crc
.00000000000000000010.checkpoint.parquet.crc
.00000000000000000010.crc.crc
.00000000000000000010.json.crc
.00000000000000000011.crc.crc
.00000000000000000011.json.crc
.00000000000000000012.crc.crc
.00000000000000000012.json.crc
.00000000000000000013.crc.crc
.00000000000000000013.json.crc
.00000000000000000014.crc.crc
.00000000000000000014.json.crc
.00000000000000000015.crc.crc
.000000000

## Delta Log

O Delta Lake mantém um log transacional no diretório `_delta_log`.
Esse log registra as alterações realizadas sobre a tabela, permitindo
rastreabilidade das operações e suporte ao controle de versões.

Abaixo é apresentado um exemplo de log associado a uma operação de
UPDATE.

In [56]:
log_v5 = spark.read.json(
    os.path.join(delta_log_path, "00000000000000000005.json")
)

log_v5.select(
    "commitInfo.operation",
    "commitInfo.operationParameters",
    "add.path",
    "remove.path"
).show(truncate=False)

+---------+----------------------------------+-------------------------------------------------------------------+-------------------------------------------------------------------+
|operation|operationParameters               |path                                                               |path                                                               |
+---------+----------------------------------+-------------------------------------------------------------------+-------------------------------------------------------------------+
|UPDATE   |{["(game_id#5555L = 8589946391)"]}|NULL                                                               |NULL                                                               |
|NULL     |NULL                              |part-00000-384b6d66-9c25-4312-bb44-fcd0bb350ed7-c000.snappy.parquet|NULL                                                               |
|NULL     |NULL                              |NULL                                   

In [57]:
log_v5.select(
    "commitInfo.operation",
    "commitInfo.operationParameters"
).show(truncate=False)

+---------+----------------------------------+
|operation|operationParameters               |
+---------+----------------------------------+
|UPDATE   |{["(game_id#5555L = 8589946391)"]}|
|NULL     |NULL                              |
|NULL     |NULL                              |
+---------+----------------------------------+



## 13. Detalhes da tabela

In [58]:
spark.sql("""
    DESCRIBE DETAIL metacritic_games
""").show(truncate=False)

[Stage 243:============================================>          (40 + 4) / 50]

+------+------------------------------------+--------------------------------------+-----------+------------------------------------------------------------------------------------------------------------------+-----------------------+-----------------------+----------------+-----------------+--------+-----------+----------+----------------+----------------+------------------------+
|format|id                                  |name                                  |description|location                                                                                                          |createdAt              |lastModified           |partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties|minReaderVersion|minWriterVersion|tableFeatures           |
+------+------------------------------------+--------------------------------------+-----------+------------------------------------------------------------------------------------------------------------------+-----------------

In [59]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(
    spark,
    "metacritic_games"
)

resultado_optimize = delta_table.optimize().executeCompaction()

resultado_optimize.show(truncate=False)

+------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|path                                                                                                              |metrics                                                                                                                                                               |
+------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|file:/home/jeanvv/Faculdade/2026-2/Engenharia de Dados/trabalho-eng-dados-apache-spark/data/delta/metacritic_games|{1, 3, {665336, 665336, 665336.0

In [60]:
spark.sql("""
    DESCRIBE HISTORY metacritic_games
""").select(
    "version",
    "timestamp",
    "operation",
    "operationParameters"
).show(truncate=False)

+-------+-----------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation|operationParameters                                                                                                                                                                      |
+-------+-----------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|30     |2026-10-07 18:29:58.422|OPTIMIZE |{predicate -> [], zOrderBy -> [], clusterBy -> [], auto -> false}                                                                                                                        |
|29     |2026-10-07 18:29:46.893|WRITE    |{mode -> Append, partitionBy -> []}  

## 14. OPTIMIZE

In [61]:
spark.sql("""
    SELECT COUNT(*) AS total_registros
    FROM metacritic_games
""").show()

spark.sql("""
    DESCRIBE DETAIL metacritic_games
""").select(
    "format",
    "numFiles",
    "sizeInBytes",
    "partitionColumns",
    "minReaderVersion",
    "minWriterVersion"
).show(truncate=False)

+---------------+
|total_registros|
+---------------+
|          21914|
+---------------+



+------+--------+-----------+----------------+----------------+----------------+
|format|numFiles|sizeInBytes|partitionColumns|minReaderVersion|minWriterVersion|
+------+--------+-----------+----------------+----------------+----------------+
|delta |1       |665336     |[]              |1               |2               |
+------+--------+-----------+----------------+----------------+----------------+



In [62]:
spark.sql("""
    SELECT
        name,
        game_id,
        metascore,
        user_score,
        rating_category
    FROM metacritic_games
    WHERE game_id IN (7777777777, 9999999999, 8589946391)
    ORDER BY game_id
""").show(truncate=False)

+---------------------+----------+---------+----------+---------------+
|name                 |game_id   |metascore|user_score|rating_category|
+---------------------+----------+---------+----------+---------------+
|Schema Evolution Game|7777777777|88.0     |8.8       |Excelente      |
+---------------------+----------+---------+----------+---------------+



In [63]:
spark.sql("""
    SELECT
        COUNT(*) AS total,
        COUNT(game_id) AS com_game_id,
        COUNT(*) - COUNT(game_id) AS sem_game_id
    FROM metacritic_games
""").show()

+-----+-----------+-----------+
|total|com_game_id|sem_game_id|
+-----+-----------+-----------+
|21914|      21914|          0|
+-----+-----------+-----------+



In [64]:
spark.sql("""
    SELECT
        name,
        platform,
        release_date,
        game_id,
        metascore,
        user_score,
        rating_category
    FROM metacritic_games
    WHERE game_id IS NULL
       OR rating_category IS NOT NULL
""").show(truncate=False)

+---------------------+--------+------------+----------+---------+----------+---------------+
|name                 |platform|release_date|game_id   |metascore|user_score|rating_category|
+---------------------+--------+------------+----------+---------+----------+---------------+
|Schema Evolution Game|PC      |2026-10-07  |7777777777|88.0     |8.8       |Excelente      |
+---------------------+--------+------------+----------+---------+----------+---------------+



In [65]:
spark.sql("""
    DELETE FROM metacritic_games
    WHERE game_id = 7777777777
""")

DataFrame[num_affected_rows: bigint]

In [66]:
spark.sql("""
    DESCRIBE metacritic_games
""").show(truncate=False)

+---------------+---------+-------+
|col_name       |data_type|comment|
+---------------+---------+-------+
|name           |string   |NULL   |
|platform       |string   |NULL   |
|release_date   |date     |NULL   |
|metascore      |double   |NULL   |
|user_score     |double   |NULL   |
|developer      |string   |NULL   |
|publisher      |string   |NULL   |
|genre          |string   |NULL   |
|game_id        |bigint   |NULL   |
|rating_category|string   |NULL   |
+---------------+---------+-------+



## 15. Validação final

In [67]:
spark.sql("""
    SELECT COUNT(*) AS total_registros
    FROM metacritic_games
""").show()

spark.sql("""
    DESCRIBE metacritic_games
""").show(truncate=False)

spark.sql("""
    DESCRIBE HISTORY metacritic_games
""").select(
    "version",
    "operation"
).show(truncate=False)

+---------------+
|total_registros|
+---------------+
|          21913|
+---------------+

+---------------+---------+-------+
|col_name       |data_type|comment|
+---------------+---------+-------+
|name           |string   |NULL   |
|platform       |string   |NULL   |
|release_date   |date     |NULL   |
|metascore      |double   |NULL   |
|user_score     |double   |NULL   |
|developer      |string   |NULL   |
|publisher      |string   |NULL   |
|genre          |string   |NULL   |
|game_id        |bigint   |NULL   |
|rating_category|string   |NULL   |
+---------------+---------+-------+

+-------+---------+
|version|operation|
+-------+---------+
|31     |DELETE   |
|30     |OPTIMIZE |
|29     |WRITE    |
|28     |DELETE   |
|27     |MERGE    |
|26     |MERGE    |
|25     |DELETE   |
|24     |UPDATE   |
|23     |WRITE    |
|22     |WRITE    |
|21     |DELETE   |
|20     |OPTIMIZE |
|19     |WRITE    |
|18     |DELETE   |
|17     |MERGE    |
|16     |MERGE    |
|15     |DELETE   |
|14 

## Validação final

Ao final das operações, a tabela `metacritic_games` contém os dados do
dataset original após a remoção dos registros duplicados.

Durante a implementação foram demonstrados os principais recursos do
Delta Lake:

- Criação e leitura de uma tabela Delta;
- Schema Enforcement;
- Schema Evolution;
- INSERT;
- UPDATE;
- DELETE;
- MERGE;
- Delta Log;
- Time Travel;
- Histórico de operações;
- OPTIMIZE para compactação dos arquivos.

A tabela foi mantida sem particionamento, pois o dataset utilizado possui
aproximadamente 22 mil registros e não apresenta volume suficiente para
justificar particionamento neste experimento.